## 0. Importaciones y parámetros

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
try:
    import ipywidgets as widgets
    from ipywidgets import interact, fixed
    has_ipywidgets = True
except ModuleNotFoundError:
    widgets = None
    interact = None
    fixed = None
%matplotlib inline

# Decremento logarítmico

## 1. Teoría (resumen)

Ecuación de movimiento para vibración libre amortiguada:
$$ m\ddot{x}(t) + c\dot{x}(t) + k x(t) = 0 $$
o en forma normalizada:
$$ \ddot{x}(t) + 2\zeta\omega_n\dot{x}(t) + \omega_n^2 x(t) = 0 $$
La solución para el caso subamortiguado ($0<\zeta<1$) es:
$$ x(t)=A e^{-\zeta\omega_n t} \cos(\omega_d t - \phi) $$
donde $\omega_d=\omega_n\sqrt{1-\zeta^2}$.

Definición de decremento logarítmico entre picos consecutivos: 
$$ \delta = \ln\left(\frac{x_n}{x_{n+1}}\right) = \frac{2\pi\zeta}{\sqrt{1-\zeta^2}}. $$
De esta relación se puede despejar $\zeta$ como:
$$ \zeta = \frac{\delta}{\sqrt{4\pi^2 + \delta^2}}. $$

## 2. Solución analítica y señal sintética

In [ ]:
# Parámetros del sistema
mass = 1.0           # kg
fn_hz = 5.0          # frecuencia natural (Hz)
omega_n = 2*np.pi*fn_hz
damping_ratio = 0.05 # ζ
omega_d = omega_n * np.sqrt(1 - damping_ratio**2)
t = np.linspace(0, 5, 5001)
A = 1.0
x = A * np.exp(-damping_ratio * omega_n * t) * np.cos(omega_d * t)
plt.figure(figsize=(8,3))
plt.plot(t, x, label=f'ζ={damping_ratio:.3f}, fn={fn_hz} Hz')
plt.xlabel('Tiempo (s)')
plt.ylabel('Desplazamiento')
plt.legend()
plt.tight_layout()

## 3. Detección de picos y cálculo de decremento 

Procedimiento:
- Detectar picos positivos en la señal sintética.
- Calcular $\delta_i = \ln(x_i/x_{i+1})$ y tomar la media sobre i para obtener $\bar{\delta}$.
- Estimar $\zeta$ a partir de $\bar{\delta}$.

In [ ]:
peaks, _ = find_peaks(x)
peak_vals = x[peaks]
# Nos quedamos con los picos positivos (si hay)
peak_vals = peak_vals[peak_vals>0]
peak_vals[:6] if peak_vals.size>6 else peak_vals

In [ ]:
log_decrements = np.log(peak_vals[:-1] / peak_vals[1:])
delta_mean = np.mean(log_decrements)
zeta_est = delta_mean / np.sqrt((2*np.pi)**2 + delta_mean**2)
print('δ medio =', delta_mean)
print('ζ estimado =', zeta_est)

## 4. Comparación numérica

A continuación se calcula el decremento logarítmico teórico a partir de la razón de amortiguamiento usada en la simulación y se compara con el valor estimado a partir de los picos.

In [ ]:
delta_theoretical = 2 * np.pi * damping_ratio / np.sqrt(1 - damping_ratio**2)
print(f'δ teórico = {delta_theoretical:.6f}')
print(f'δ estimado = {delta_mean:.6f}')
print(f'Error relativo δ = {abs(delta_mean - delta_theoretical)/delta_theoretical*100:.3f} %')
print(f'ζ real = {damping_ratio:.6f}')
print(f'ζ estimado = {zeta_est:.6f}')

## 5. Comparación y comentarios

Se espera que la estimación converja al valor real de $\zeta$ para señales con relación señal/ruido razonable y suficientes picos detectados. En presencia de ruido puede ser necesario filtrar la señal o usar ajuste exponencial sobre envolvente de picos.

## 6. Sección interactiva

A continuación se presenta una visualización interactiva que permite ajustar parámetros básicos del sistema y observar la respuesta temporal con los picos correspondientes.

In [ ]:
def compute_damped_response(amplitude, fn_hz, damping_ratio, duration=5.0, n_points=5001):
    omega_n = 2 * np.pi * fn_hz
    omega_d = omega_n * np.sqrt(max(0.0, 1 - damping_ratio**2))
    t = np.linspace(0, duration, n_points)
    x = amplitude * np.exp(-damping_ratio * omega_n * t) * np.cos(omega_d * t)
    peaks, _ = find_peaks(x)
    peak_vals = x[peaks]
    peak_vals = peak_vals[peak_vals > 0]
    if peak_vals.size > 1:
        log_decrements = np.log(peak_vals[:-1] / peak_vals[1:])
        delta_mean = np.mean(log_decrements)
        zeta_est = delta_mean / np.sqrt((2*np.pi)**2 + delta_mean**2)
    else:
        delta_mean = np.nan
        zeta_est = np.nan
    return t, x, peaks, peak_vals, delta_mean, zeta_est

In [ ]:
def plot_interactive_response(amplitude, fn_hz, damping_ratio):
    t, x, peaks, peak_vals, delta_mean, zeta_est = compute_damped_response(amplitude, fn_hz, damping_ratio)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, x, label='Respuesta amortiguada')
    ax.plot(t[peaks], x[peaks], 'ro', label='Picos detectados')
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('Desplazamiento')
    ax.set_title(f'ζ estimado: {zeta_est:.4f} / ζ real: {damping_ratio:.4f}')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'δ medio = {delta_mean:.4f}')
    print(f'ζ estimado = {zeta_est:.4f}')

In [ ]:
if not has_ipywidgets:
    print('ipywidgets no está instalado. Instale con `pip install ipywidgets` para ejecutar esta sección.')
else:
    interact(
        plot_interactive_response,
        amplitude=widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description='Amplitud'),
        fn_hz=widgets.FloatSlider(value=5.0, min=1.0, max=10.0, step=0.5, description='fn (Hz)'),
        damping_ratio=widgets.FloatSlider(value=0.05, min=0.01, max=0.25, step=0.01, description='ζ'),
    )